In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_candidates = [Path("census.csv"), Path("Machine Learning Project 2/census.csv")]
data_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("census.csv")
df_original = pd.read_csv(data_path)
df = df_original.copy(deep=True)
print("dataset_path", data_path)
print("loaded", df.shape)

In [ ]:
display(df.head())
print("shape", df.shape)
print("columns", df.columns.tolist())
df.info()
display(df.dtypes.astype(str).to_frame("dtype"))

In [ ]:
display(df.describe().T)
display(df.describe(include=["object"]).T)

In [ ]:
income_counts = df["income"].value_counts(dropna=False)
income_proportions = df["income"].value_counts(normalize=True, dropna=False)
display(income_counts.to_frame("count"))
display(income_proportions.to_frame("proportion"))

In [ ]:
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
missing_counts = df.isna().sum()
question_mark_counts = {column: int(df[column].astype("string").str.strip().eq("?").sum()) for column in text_columns}
whitespace_only_counts = {column: int(df[column].astype("string").str.fullmatch(r"\s*").fillna(False).sum()) for column in text_columns}
display(missing_counts.to_frame("missing_count"))
display(pd.Series(question_mark_counts, name="question_mark_count").to_frame())
display(pd.Series(whitespace_only_counts, name="whitespace_only_count").to_frame())

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("duplicate_rows", duplicate_count)

In [ ]:
expected_numeric_columns = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
numeric_conversion_errors = {}
for column in expected_numeric_columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    numeric_conversion_errors[column] = int((converted.isna() & df[column].notna()).sum())
expected_columns = ["age", "workclass", "education_level", "education-num", "marital-status", "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
print("numeric_conversion_errors", numeric_conversion_errors)
print("unexpected_columns", [column for column in df.columns if column not in expected_columns])
print("missing_expected_columns", [column for column in expected_columns if column not in df.columns])
display(df.dtypes.astype(str).to_frame("dtype"))

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()
outer_whitespace_counts = {column: int(df[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) for column in categorical_columns}
case_variants = {}
for column in categorical_columns:
    values = df[column].dropna().astype(str).unique().tolist()
    groups = {}
    for value in values:
        groups.setdefault(value.strip().casefold(), set()).add(value.strip())
    variants = {key: sorted(group) for key, group in groups.items() if len(group) > 1}
    if variants:
        case_variants[column] = variants
print("outer_whitespace_counts", outer_whitespace_counts)
print("case_variants", case_variants)
for column in categorical_columns:
    print(column, "unique_values", int(df[column].nunique(dropna=False)))
    display(df[column].value_counts(dropna=False).to_frame("count"))

In [ ]:
valid_numeric_ranges = {"age": (0, 120), "education-num": (1, 16), "capital-gain": (0, None), "capital-loss": (0, None), "hours-per-week": (0, 168)}
invalid_numeric_counts = {}
for column, (lower, upper) in valid_numeric_ranges.items():
    values = pd.to_numeric(df[column], errors="coerce")
    invalid = values.notna()
    if lower is not None:
        invalid &= values < lower
    if upper is not None:
        invalid |= values.notna() & (values > upper)
    invalid_numeric_counts[column] = int(invalid.sum())
valid_income_values = {"<=50K", ">50K"}
invalid_income_values = sorted(set(df["income"].dropna().astype(str).str.strip()) - valid_income_values)
print("invalid_numeric_counts", invalid_numeric_counts)
print("invalid_income_values", invalid_income_values)
print("zero_capital_gain", int(df["capital-gain"].eq(0).sum()))
print("zero_capital_loss", int(df["capital-loss"].eq(0).sum()))

In [ ]:
outlier_results = {}
for column in df.select_dtypes(include=np.number).columns:
    values = df[column].dropna()
    first_quartile = values.quantile(0.25)
    third_quartile = values.quantile(0.75)
    interquartile_range = third_quartile - first_quartile
    lower_bound = first_quartile - 1.5 * interquartile_range
    upper_bound = third_quartile + 1.5 * interquartile_range
    outlier_count = int(((values < lower_bound) | (values > upper_bound)).sum())
    outlier_results[column] = {"lower_bound": float(lower_bound), "upper_bound": float(upper_bound), "outlier_count": outlier_count}
display(pd.DataFrame.from_dict(outlier_results, orient="index"))

In [ ]:
df_clean = df.copy(deep=True)
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    normalized_values = df_clean[column].astype("string").str.strip()
    df_clean[column] = normalized_values.mask(normalized_values.eq("") | normalized_values.eq("?"), pd.NA)
target_missing_count = int(df_clean["income"].isna().sum())
if target_missing_count:
    df_clean = df_clean.loc[df_clean["income"].notna()].copy()
for column in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[column].isna().any():
        median_value = df_clean[column].median()
        if pd.isna(median_value):
            raise ValueError(f"No observed values available to impute {column}")
        df_clean[column] = df_clean[column].fillna(median_value)
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    if df_clean[column].isna().any():
        modes = df_clean[column].mode(dropna=True)
        if modes.empty:
            raise ValueError(f"No observed category available to impute {column}")
        df_clean[column] = df_clean[column].fillna(modes.iloc[0])
print("target_rows_removed_for_missing_target", target_missing_count)
print("remaining_missing_values", int(df_clean.isna().sum().sum()))

In [ ]:
identifier_columns = [column for column in df_clean.columns if column.casefold() == "id" or column.casefold().endswith("_id")]
duplicate_count_after_missing_handling = int(df_clean.duplicated().sum())
print("identifier_columns", identifier_columns)
print("duplicate_rows_reviewed", duplicate_count_after_missing_handling)
print("duplicate_rows_removed", 0)
print("duplicate_rows_retained", duplicate_count_after_missing_handling)

In [ ]:
for column in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()
print("outer_whitespace_remaining", {column: int(df_clean[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) for column in df_clean.select_dtypes(include=["object", "string"]).columns})
print("income_values", sorted(df_clean["income"].dropna().unique().tolist()))

In [ ]:
if not set(df_clean["income"].dropna().unique()).issubset({"<=50K", ">50K"}):
    raise ValueError("Unexpected income values need review before cleaning can continue")
invalid_replacements = {}
for column, (lower, upper) in valid_numeric_ranges.items():
    values = pd.to_numeric(df_clean[column], errors="coerce")
    invalid = values.notna()
    if lower is not None:
        invalid &= values < lower
    if upper is not None:
        invalid |= values.notna() & (values > upper)
    invalid_replacements[column] = int(invalid.sum())
    if invalid.any():
        df_clean.loc[invalid, column] = np.nan
        median_value = pd.to_numeric(df_clean[column], errors="coerce").median()
        if pd.isna(median_value):
            raise ValueError(f"No valid values available to replace invalid values in {column}")
        df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce").fillna(median_value)
print("impossible_numeric_values_replaced", invalid_replacements)

In [ ]:
print("iqr_outliers_reviewed", {column: values["outlier_count"] for column, values in outlier_results.items()})
print("outliers_removed", 0)
print("outliers_capped_or_clipped", 0)

In [ ]:
df_clean.columns = df_clean.columns.str.strip()
if df_clean.columns.duplicated().any():
    raise ValueError("Duplicate column names after trimming")
for column in expected_numeric_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="raise")
print("columns", df_clean.columns.tolist())
print("dtypes", df_clean.dtypes.astype(str).to_dict())
print("shape", df_clean.shape)

In [ ]:
comparison = pd.DataFrame({"original_dtype": df_original.dtypes.astype(str), "cleaned_dtype": df_clean.dtypes.astype(str), "original_missing": df_original.isna().sum(), "cleaned_missing": df_clean.isna().sum()})
display(comparison)
print("original_shape", df_original.shape)
print("cleaned_shape", df_clean.shape)
print("original_duplicate_rows", int(df_original.duplicated().sum()))
print("cleaned_duplicate_rows", int(df_clean.duplicated().sum()))
print("original_missing_values", int(df_original.isna().sum().sum()))
print("cleaned_missing_values", int(df_clean.isna().sum().sum()))

In [ ]:
assert df_clean.columns.tolist() == df_original.columns.tolist()
assert not df_clean.isna().any().any()
assert not df_clean.columns.duplicated().any()
assert set(df_clean["income"].unique()).issubset({"<=50K", ">50K"})
assert all(int(df_clean[column].map(lambda value: isinstance(value, str) and value != value.strip()).sum()) == 0 for column in df_clean.select_dtypes(include=["object", "string"]).columns)
for column, (lower, upper) in valid_numeric_ranges.items():
    assert (lower is None or df_clean[column].ge(lower).all())
    assert (upper is None or df_clean[column].le(upper).all())
print("final_shape", df_clean.shape)
print("final_missing_values", int(df_clean.isna().sum().sum()))
print("final_duplicate_rows", int(df_clean.duplicated().sum()))
print("final_income_values", sorted(df_clean["income"].unique().tolist()))

In [ ]:
cleaned_path = data_path.with_name("census_cleaned.csv")
if cleaned_path.exists():
    existing_cleaned = pd.read_csv(cleaned_path)
    pd.testing.assert_frame_equal(df_clean, existing_cleaned, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
    print("existing_cleaned_file_matches", cleaned_path)
else:
    df_clean.to_csv(cleaned_path, index=False)
    print("saved_path", cleaned_path)
print("saved_shape", pd.read_csv(cleaned_path).shape)